# Global Weather Forecasts and Wind Particles with geeViz

Forecast weather from three global models, with an animated wind field
in the style of windy.com.

| Model | `model` | Collection | Resolution | Reaches forward |
|---|---|---|---|---|
| NOAA GFS | `gfs` | `NOAA/GFS0P25` | 0.25° | ~16 days |
| ECMWF IFS | `euro` | `ECMWF/NRT_FORECAST/IFS/OPER` | ~0.4° | ~6 days |
| WeatherNext 3 | `weathernext` | `.../weathernext_3_0_0_0p1deg` | 0.1° | ~15 days |

`geeViz.weather` puts all three behind one call. Verified against the
live collections on 2026-09-10.

Each model marks time differently — GFS and ECMWF use
`creation_time` / `forecast_time` as epoch milliseconds, WeatherNext
uses `start_time` / `end_time` as ISO 8601 strings. Reconciling that is
what `getForecastData` is for, so you can ask for a date range and get
the right images regardless of model.

Copyright 2026 Ian Housman

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

[![github](https://img.shields.io/badge/-see%20sources-white?logo=github&labelColor=555)](https://github.com/gee-community/geeviz/blob/master/examples/weather_forecast_examples.ipynb)
[![github](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gee-community/geeViz/blob/master/examples/weather_forecast_examples.ipynb)

## Setup

In [ ]:
import datetime

try:
    import geeViz.geeView as gv
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'geeViz'])
    import geeViz.geeView as gv

import geeViz.weather as wx

ee = gv.ee
Map = gv.Map
Map.clearMap()
Map.setQueryDateFormat('YYYY-MM-dd HH:mm')

now = datetime.datetime.now(datetime.timezone.utc)
fut_start = (now + datetime.timedelta(days=1)).strftime('%Y-%m-%dT%H:%MZ')
fut_end   = (now + datetime.timedelta(days=3)).strftime('%Y-%m-%dT%H:%MZ')
# Hurricane Helene (2024) - made landfall: September 23, 2024
past_start = '2024-09-26T00:00Z'
past_end   = '2024-09-27T23:59Z'

study_area = ee.Geometry.Rectangle([-125, 31, -102, 49])   # US West
Map.centerObject(study_area, 5)


# WeatherNext is a gated dataset. Check access up front so an unentitled
# account gets a request-access URL rather than a cryptic error deep in
# the pipeline. GFS and ECMWF are open and always run.
HAS_WN = False
try:
    ee.ImageCollection(wx.MODELS['weathernext']['collection']).limit(1).size().getInfo()
    HAS_WN = True
except Exception as e:
    print(f'[preflight] WeatherNext not readable: {type(e).__name__}')
    print('           Request access: https://developers.google.com/weathernext/guides/earth-engine')

print(f'forward {fut_start}..{fut_end}   past {past_start}..{past_end}   WeatherNext={HAS_WN}')

## Picking the right images

`getForecastData(startDate, endDate, model)` chooses based on where the
window sits relative to now:

| window | what you get |
|---|---|
| **past** | the shortest-lead image from every run initialized in the window — one fresh analysis per cycle, the best record of what actually happened |
| **future** | a single run: the most recent one that REACHES the end of your window, filtered to it |
| **spanning now** | analyses up to the most recent initialization, then that run's forecast from there — merged |

"Shortest lead" rather than literally lead 0: GFS and ECMWF publish a
0-hour analysis, WeatherNext's `forecast_hour` runs 1..360 and never
reaches 0. The value is read from each collection rather than assumed.

The lead hours below are the proof.

In [ ]:
# Every frame's system:time_start, so you can see it is the VALID time.
# WeatherNext natively carries the run's INIT time there -- one value for
# a whole run -- so a time lapse built on the raw collection collapses to
# a single frame. getForecastData restamps it on the way out, for every
# model, and these are the numbers that show it.
def frames(ic, label, show=4):
    d = ee.Dictionary({
        'n': ic.size(),
        'n_stamps': ic.aggregate_count_distinct('system:time_start'),
        'sts': ic.aggregate_array('system:time_start').sort(),
        'valid': ic.aggregate_array('valid_time').sort(),
        'leads': ic.aggregate_array('lead_hours').sort(),
    }).getInfo()
    utc = lambda ms: datetime.datetime.fromtimestamp(
        ms / 1000, datetime.timezone.utc).strftime('%Y-%m-%d %H:%M')
    stamps = [utc(t) for t in d['sts']]
    if len(stamps) <= 2 * show:
        line = ' | '.join(stamps)          # short enough to show whole
    else:
        line = (' | '.join(stamps[:show]) + '  ...  '
                + ' | '.join(stamps[-show:]))
    print(f"{label}: n={d['n']}, {d['n_stamps']} distinct system:time_start")
    print(f"    {line}")
    print(f"    system:time_start == valid_time: {d['sts'] == d['valid']}"
          f"   one stamp per image: {d['n_stamps'] == d['n']}"
          f"   leads {d['leads'][0]}..{d['leads'][-1]}")

gfs_fut  = wx.getForecastData(ee.Date(fut_start), ee.Date(fut_end), 'gfs')
gfs_past = wx.getForecastData(past_start, past_end, 'gfs')

frames(gfs_fut,  'gfs future ')   # one run, leads climbing
frames(gfs_past, 'gfs past   ')   # analyses only, one lead

# Hurricane Helene is September 2024, which only GFS reaches: ECMWF NRT
# begins 2024-11-12 and WeatherNext 3 begins in 2026. Those two return
# the empty-window sentinel for this range -- one fully masked image
# carrying lead_hours = -1, so downstream code cannot die on .first().
for m in ('euro', 'weathernext'):
    if m == 'weathernext' and not HAS_WN:
        continue
    ic = wx.getForecastData(past_start, past_end, m)
    d = ee.Dictionary({'n': ic.size(),
                       'lead': ic.aggregate_min('lead_hours')}).getInfo()
    tag = 'EMPTY (sentinel)' if d['lead'] == -1 else 'data'
    print(f"{m:12s} n={d['n']:3d} lead={d['lead']:4d}  {tag}"
          f"   <- archive does not go back to {past_start}")


### A window that spans now

Ask for yesterday through three days out and you get both halves,
seamed at the most recent initialization: shortest-lead analyses up to
that moment, then that single run's forecast onward. The lead hours show
the join — they start at the analysis lead and climb through a real
forecast horizon.


In [ ]:
span_start = (now - datetime.timedelta(days=2)).strftime('%Y-%m-%d')
span_end   = (now + datetime.timedelta(days=3)).strftime('%Y-%m-%d')

for m in (['gfs', 'euro'] + (['weathernext'] if HAS_WN else [])):
    frames(wx.getForecastData(span_start, span_end, m), f'{m:12s}')

print('')
print(f'asked {span_start} .. {span_end}  (now is {now:%m-%d %H:%M} UTC)')
print('The stamps run continuously across the seam: the analyses and the')
print('forecast run are different sources, but one ordered time axis.')


## Wind: a speed raster plus animated particles

`Map.addWindLayer` adds **two** layers, the way windy.com does — a
smooth speed raster carrying the reading, with particle trails over it
showing the flow:

1. **`… speed`** — bicubic raster, and the layer a click reads. It
   carries both `speed` and `direction`.
2. **`… particles`** — an animated canvas. The wind components are
   decoded from Earth Engine PNG tiles (u in red, v in green, stretched
   ±40 m/s), so the particles follow the map **anywhere you pan**
   rather than being confined to a region fixed up front.

`viz` follows `addLayer`'s conventions. `bands` defaults to the image's
**first two bands in order** (dx, dy) — by position, because every
product names its components differently.

In [ ]:
# One instant, not an average: the vector mean of a veering wind is not
# a wind anyone experiences.
Map.clearMap()
wind_img = ee.Image(gfs_past.sort('valid_time',True).first())
date = wind_img.date().format('YYYY-MM-dd HH:mm').getInfo()   # Joda: mm is minutes, MM is month
speed_dir, tiles = Map.addWindLayer(wind_img, {
    # ---- the raster half: what a click reads -------------------------
    'units': 'km/hr',             # 'm/s' | 'km/hr' | 'mi/hr' | 'kt'. default 'km/hr'
                                  #   'kt' is what METAR, TAF, marine forecasts
                                  #   and windy.com report wind in. NOT mi/hr --
                                  #   a knot is a NAUTICAL mile per hour.
    'bands': ['u', 'v'],          # the dx/dy components; default = first two, in order
    'min': 0, 'max': 100,         # speed stretch. default max FOLLOWS THE UNIT
                                  #   (15 m/s | 54 km/hr | 34 mi/hr | 30 kt), so switching
                                  #   units cannot leave the raster one flat colour
    'palette': list(wx.WIND_PALETTE),   # default: windy.com's own ramp, 0-30 m/s
    'directionConvention': 'from',      # 'from' (default, meteorological: 270 is a
                                        #   westerly) | 'to' (where the air is going)

    # ---- colour ------------------------------------------------------
    'particleColor': '#ff0',      # any CSS hex.  default '#fff'.  Dark basemaps
                                  #   want white; over pale terrain try '#222'
    'particleOpacity': 0.9,       # alpha at the HEAD. 0-1, default 0.9
                                  #   low 0.4 = ghostly | high 1.0 = hard-edged

    # ---- width (across the streak; LENGTH along it is trailLength) ----
    'particleStrokeWeight': 1.1,  # base width in px. default 1.1
                                  #   min/max below default to 0.45x and 1.5x of it,
                                  #   so changing ONLY this rescales the whole taper
    'particleMinWidth': 1.1 * 0.45,  # width at the TAIL, px. default strokeWeight*0.45
                                    #   low 0.2 = hairline tail | high = blunt ribbon
    'particleMaxWidth': 1.1 * 1.5,   # width at the HEAD, px. default strokeWeight*1.5
                                    #   set equal to MinWidth for a constant-width
                                    #   ribbon instead of a comet

    # ---- shape -------------------------------------------------------
    'particleTrailLength': 13,    # frames of history drawn. default 13
                                  #   THIS x the per-frame step IS the streak length
                                  #   low 8 = short dashes | high 60 = long ribbons
                                  #   (cost is linear: 60 draws ~2.3x the segments)
    'particleTaper': 2.1,         # exponent on the tail fade. default 2.1
                                  #   1.0 = linear wedge | >2 stretches the faint
                                  #   part out, which is what reads as a comet
    'particleHeadBoost': 1.6,     # alpha multiplier on the leading segment. default 1.6
                                  #   1.0 = no bright tip | 2+ = a hard spark
    'particleLineCap': 'round',   # 'round' (default, tapered tips) | 'butt' (blunt)

    # ---- speed -------------------------------------------------------
    'particleSpeed': 0.5,        # PIXELS PER FRAME for each m/s of wind, at the
                                  #   equator. default 0.5.
                                  #   Sets how fast the field moves AND, since
                                  #   length is proportional to it, how long the
                                  #   streaks are.  low 0.12 = slow, short |
                                  #   high 0.7 = fast, long.
                                  #   Zoom does not enter into it: a streak is the
                                  #   same size on screen however far you zoom.
    # The apparent-speed FLOOR and CEILING are not separate knobs: they
    # are 'min' and 'max' above, converted from 'units' to m/s. Streaks
    # start and stop growing exactly where the colour ramp does -- past
    # 'max' the raster is one flat colour, and a longer streak there
    # would claim a difference the map has stopped showing.
    #   here: min 0 -> 1.0 m/s, max 100 mi/hr -> 44.7 m/s
    # A stretch starting at 0 (nearly every wind map) would leave no
    # floor at all, so 0 becomes 1 m/s -- length is proportional to
    # speed, and without a floor a light breeze is a one-pixel dot.
    # There are no separate floor/ceiling parameters: widen 'min'/'max'
    # to widen the range the streaks respond over.
    #
    # Both apply to the ADVECTION only. Direction is untouched and the
    # raster and click query still report the true value, so never read
    # a wind speed off a streak length.

    # ---- lifetime ----------------------------------------------------
    'particleMinAge': 11.25,      # shortest lifetime, frames. default maxAge * 0.25
    'particleMaxAge': 45,         # longest lifetime, frames. default 45
                                  #   each particle draws its OWN lifetime from this
                                  #   range, which is what puts short streaks
                                  #   alongside long ones. Set equal for a uniform
                                  #   comb.  At the end of its life a particle keeps
                                  #   flying while its trail RETRACTS, so the streak
                                  #   shortens away instead of blinking out.
                                  #   NOTE speed, trailLength, maxAge and the
                                  #   renderer's 30fps cap are ONE group: length is
                                  #   trail x speed and motion is speed x fps, so at
                                  #   a fixed rate you cannot shorten the trail
                                  #   without speeding the field up.

    # ---- layout ------------------------------------------------------
    'particleLayout': 'random',   # where particles START. default 'random'
                                  #   'random'     = scattered. What a flow field
                                  #                  usually wants: the eye reads
                                  #                  the streaks, not the origins.
                                  #   'grid'       = strict lattice, even coverage,
                                  #                  the way a barb plot is laid out
                                  #   'randomGrid' = that lattice under ONE random
                                  #                  offset, so spacing stays even
                                  #                  but the rows do not land in the
                                  #                  same place every time
                                  #   Lattice particles respawn in their own cell,
                                  #   so the pattern does not erode into noise.

    # ---- count -------------------------------------------------------
    'particleDensity': 1.2,      # particles per pixel of CANVAS WIDTH. default 1.2
                                  #   -> ~2040 on a 1700px canvas, ~2300 at 1920px.
                                  #   low 0.8 = sparse and cheap | high 4 = dense.
                                  #   This is the main cost knob. No floor or ceiling:
                                  #   width x density is the right answer at any size,
                                  #   and it does not vary with zoom.
    # 'particleCount': 3000,      # overrides the density derivation outright.
                                  #   Omitted by default so the canvas is measured.
}, name=f'GFS 10 m wind {date}')

# Units drive the stretch: a 0..15 scale read as km/h paints the map one
# flat colour, so max defaults per unit (15 m/s, 54 km/h, 34 mi/h).
print({u: wx.DEFAULT_MAX_SPEED[u] for u in wx.SPEED_UNITS})
Map.setCenter(-84,28.4,7)
Map.view(True)

### What a click reports

Direction is meteorological by default — the bearing the wind blows
FROM, so 270 is a westerly, which is what forecast products and barb
charts mean. Pass `'directionConvention': 'to'` for the way the air is
moving, which is what a spread model wants.

In [ ]:
denver = ee.Geometry.Point([-104.99, 39.74])
print(speed_dir.reduceRegion(ee.Reducer.first(), denver, 27830).getInfo())

## The other models

Same call, different image.

In [ ]:
Map.clearMap()

# Pick ONE valid time and give every model the frame nearest to it.
#
# Taking each model's own first frame instead is the obvious thing and
# it quietly compares different hours: ECMWF steps 3-hourly and
# WeatherNext hourly, so their first frames in the same window can sit
# an hour or more apart. A model-vs-model difference and a
# one-hour-later difference look identical on a map.
#
# (The same trap catches a geeViz/windy.com comparison, where windy
# shows LOCAL time and these labels are UTC.)
_t = ee.Date(fut_start).advance(12, 'hour').millis()
# Snapped to a 3-hour UTC boundary: ECMWF publishes only every 3 hours,
# so an off-grid target guarantees an hour of mismatch no matter how
# well you choose the nearest frame.
_step = 3 * 3600 * 1000
target = ee.Date(_t.divide(_step).round().multiply(_step))

def nearest(model):
    ic = wx.getForecastData(fut_start, fut_end, model)
    ic = ic.map(lambda i: i.set(
        'dt', ee.Number(i.get('valid_time')).subtract(target.millis()).abs()))
    return ee.Image(ic.sort('dt').first())

MODEL_VIZ = {'units': 'kt', 'min': 0, 'max': 60}   # windy's own scale

euro_fut = nearest('euro')
euro_date = euro_fut.date().format('YYYY-MM-dd HH:mm').getInfo()
Map.addWindLayer(euro_fut, {**MODEL_VIZ, 'particleColor': '#ffe066'},
                 name=f'ECMWF wind {euro_date}', visible=True)

if HAS_WN:
    wn_fut = nearest('weathernext')
    wn_date = wn_fut.date().format('YYYY-MM-dd HH:mm').getInfo()
    Map.addWindLayer(wn_fut, {**MODEL_VIZ, 'particleColor': '#8ecae6'},
                     name=f'WeatherNext 3 wind {wn_date}', visible=False)
    print(f'target {target.format("YYYY-MM-dd HH:mm").getInfo()}  ->  '
          f'ECMWF {euro_date}   WeatherNext {wn_date}')

# Two particle layers at once is roughly twice the per-frame drawing
# work, on one shared 30 fps budget -- drop particleDensity to ~0.6 on
# each if the animation stutters.
Map.turnOnInspector()
Map.view(True)

## Variables across models — and the units trap

Three things are the same whichever model produced the image, and that is
what makes two models subtractable:

| | |
|---|---|
| **time** | `system:time_start` is the **valid** time — not the run time, which is what WeatherNext stamps natively |
| **name** | the band is the `wx.VARIABLES` key, not the product's spelling of it |
| **unit** | whatever `wx.CANONICAL_UNITS` says, and the image carries a `wx_units` property saying so |

The raw products disagree on every one. **GFS and ECMWF publish 2 m
temperature in Celsius; WeatherNext publishes Kelvin** — at Denver for
one hour: 29.46 / 27.46 / 297.40. Charting those raw puts one line 273
units off the others, which reads as a model blow-up rather than a unit
mismatch. Cloud cover is a fraction in one product and a percent in
another. Pressure is Pascals everywhere and hectopascals nowhere.

Where a model does not publish a variable it raises, rather than
returning an empty layer — ECMWF and WeatherNext publish dewpoint
instead of relative humidity, for instance.

`variable=None` is the one path that does **not** normalize: with no
table entry there is nothing that says what the raw bands are in.

In [ ]:
import json as _json
for name, entry in wx.VARIABLES.items():
    # An entry is a (band, unit) tuple when the model publishes it in a
    # usable form. Anything else means it does not -- including a string,
    # which is a REASON. Test the type, not truthiness: a non-empty
    # string is truthy, and entry[m][0] on one gives you a letter.
    have = {m: (entry[m][0] if wx.publishes(name, m) else None)
            for m in ('euro', 'gfs', 'weathernext')}
    print(f'{name:26s} -> {wx.CANONICAL_UNITS[name]:6s} {_json.dumps(have)}')

print()
print('ECMWF precipitation is the interesting one:')
try:
    wx.getForecastData(fut_start, fut_end, 'euro', variable='precipitation')
except ValueError as e:
    print(f'  {e}')

try:
    wx.getForecastData(fut_start, fut_end, 'euro',
                       variable='relative_humidity_2m')
except ValueError as e:
    print(f'\neuro relative humidity -> {e}')

In [ ]:
# One call per variable. getForecastData does the run selection
# too -- a forward window comes from the single most recent run,
# so the field does not jump where two runs disagree.
#
# wx ships windy.com's own ramps, so a geeViz map and a windy map of
# the same hour read the same way.
Map.clearMap()
TEMP_VIZ = {'min': -5, 'max': 40,
            'palette': list(wx.TEMPERATURE_PALETTE),
            'yLabel': 'Temperature (C)'}

# One frame, not a mosaic. Mosaicking a forward window collapses
# every hour of it into whichever frame happens to win per pixel, and
# the result has no single valid time to put in the name -- so the
# layer says "temperature" and quietly means "some hour or other".
def first_frame(model, variable):
    img = ee.Image(wx.getForecastData(fut_start, fut_end, model,
                                      variable=variable)
                     .sort('valid_time').first())
    # Joda: mm is minutes, MM is month.
    return img, img.date().format('YYYY-MM-dd HH:mm').getInfo()

for model in ('gfs', 'euro'):
    t, t_date = first_frame(model, 'temperature_2m')
    Map.addLayer(t.clip(study_area), TEMP_VIZ,
                 f'{model.upper()}: Temperature (2 m) {t_date}',
                 model == 'gfs')

rh, rh_date = first_frame('gfs', 'relative_humidity_2m')
Map.addLayer(rh.clip(study_area),
             {'min': 0, 'max': 100, 'palette': 'd73027,ffffbf,4575b4',
              'yLabel': 'Relative humidity (%)'},
             f'GFS: Relative humidity (2 m) {rh_date}', False)

precip, p_date = first_frame('gfs', 'precipitation')
Map.addLayer(precip.clip(study_area),
             {'min': 0, 'max': 5,
              'palette': list(wx.PRECIP_PALETTE),
              'yLabel': 'Precipitation (mm/hr)'},
             f'GFS: Precipitation {p_date}', False)

Map.turnOnInspector()
Map.view(True)

> **GFS bands are not stable across the collection**, and the split is
> by AGE. The oldest images carry `total_precipitation_surface`; every
> image of the last few days carries `precipitation_rate` alongside
> `gust`, `haines_index` and `ventilation_rate`. An audit that samples
> `.first()` of the unfiltered collection therefore reports the names in
> `wx.VARIABLES` as broken — they are correct for recent data, which is
> what a forecast request asks for. If a select fails, inspect an image
> from the window you actually want.

## WeatherNext ensemble percentiles

The 64 members are published **pre-aggregated** as `_mean`, `_p10`,
`_p25`, `_p50`, `_p75`, `_p90`. So forecast spread is the difference of
two of these, not a reduction over members — which is both cheaper and
the only option, since the members themselves are not in the collection.

In [ ]:
Map.clearMap()
if HAS_WN:
    def pct(stat):
        return ee.Image(wx.getForecastData(fut_start, fut_end, 'weathernext',
                                           variable='temperature_2m',
                                           stat=stat)
                          .sort('valid_time').first())

    t10, t90 = pct('p10'), pct('p90')
    sp_date = t90.date().format('YYYY-MM-dd HH:mm').getInfo()
    spread = t90.subtract(t10).rename('spread')
    Map.addLayer(spread.clip(study_area),
                 {'min': 0, 'max': 10,
                  'palette': '000004,420a68,932667,dd513a,fca50a,fcffa4',
                  'yLabel': 'p90 - p10 (C)'},
                 f'WeatherNext 3: temperature spread {sp_date}', True)
    print('spread = p90 - p10; widens with lead time as confidence drops')

Map.turnOnInspector()
Map.view(True)

## Terrain downscaling

A forecast wind field is 0.25 degrees for GFS — about 28 km, an order of
magnitude coarser than the terrain that actually steers surface wind. Over
mountains the map shows one value across a whole range.

`wx.downscaleWind` applies the MicroMet topographic adjustment
([Liston & Elder 2006](https://journals.ametsoc.org/view/journals/hydr/7/2/jhm486_1.xml)),
the standard intermediate-complexity method: speed is scaled by terrain
**slope in the wind direction** and **curvature**, and the flow is turned
somewhat along the topography.

| term | effect |
|---|---|
| windward slope | faster |
| lee slope | slower |
| ridge (convex) | faster |
| valley (concave) | slower |

**It is not a wind model.** No mass conservation, no momentum — a solver
like [WindNinja](https://www.firelab.org/project/windninja) does that and
cannot be expressed as a per-pixel Earth Engine computation. What you get is
the first-order terrain signal. Read it as a better-resolved rendering of the
same forecast, not as new information about the atmosphere.

The adjustment is bounded by construction: speed to 0.5–1.5x the input and
direction to ±14.3°. A downscaler free to double the wind would be
asserting something the forecast never said.

In [ ]:
# Colorado Rockies: real relief, and the Helene frame is the wrong place
# for terrain (the Gulf is flat).
rockies = ee.Geometry.Rectangle([-107.5, 38.5, -105.0, 40.5])

# One instant of GFS wind over the mountains.
mtn = ee.Image(wx.getForecastData(past_start, past_end, 'gfs').first())

# 500 m. `region` is required and is not a formality: the slope and
# curvature terms are normalised by the strongest terrain in the domain,
# so the same mountain downscales differently inside a small box than
# inside a continental one. That is how the method is defined.
mtn_ds = wx.downscaleWind(mtn, region=rockies, scale=500)

# What it did, measured rather than asserted.
def spread(img, label):
    mag = img.select(0).hypot(img.select(1))
    d = mag.reduceRegion(ee.Reducer.percentile([2, 50, 98]), rockies, 500,
                         bestEffort=True, maxPixels=1e9).getInfo()
    # Key by SUFFIX, not sort order: sorted() gives p2, p50, p98 in
    # alphabetical order, which is not percentile order, and the labels
    # end up on the wrong numbers.
    g = {k.rsplit('_p', 1)[1]: v for k, v in d.items()}
    print(f'{label:14s} p2={g["2"]:5.2f}  p50={g["50"]:5.2f}  p98={g["98"]:5.2f} m/s')

spread(mtn, 'coarse')
spread(mtn_ds, 'downscaled')

# Percentiles of SPEED are a poor witness for what the adjustment did.
# The coarse field already spans 0.01-5.3 m/s across the Rockies from the
# synoptic gradient alone, while the terrain term redistributes WITHIN
# each 28 km cell -- the two mix, and domain p2/p98 move by a couple of
# percent even when the per-pixel adjustment is large. The RATIO isolates
# it, and the ratio is the thing downscaleWind actually computes.
ratio = (mtn_ds.select(0).hypot(mtn_ds.select(1))
         .divide(mtn.select(0).hypot(mtn.select(1))))
r = ratio.reduceRegion(ee.Reducer.percentile([2, 50, 98]), rockies, 500,
                       bestEffort=True, maxPixels=1e9).getInfo()
g = {k.rsplit('_p', 1)[1]: v for k, v in r.items()}
print()
print(f'terrain adjustment  p2={g["2"]:.2f}x  p50={g["50"]:.2f}x  '
      f'p98={g["98"]:.2f}x')
print()
print('Valleys slower, ridges faster -- by up to ~30% here, inside the')
print('0.5-1.5x the method allows. The median sits at 1.00: this')
print('redistributes the forecast, it does not add or remove wind.')

Map.clearMap()
mtn_date = mtn.date().format('YYYY-MM-dd HH:mm').getInfo()
VIZ = {'units': 'mi/hr', 'min': 0, 'max': 60, 'particleDensity': 1.2}
Map.addWindLayer(mtn,    VIZ,
                 f'GFS 10 m wind {mtn_date} - coarse (28 km)', False)
Map.addWindLayer(mtn_ds, VIZ,
                 f'GFS 10 m wind {mtn_date} - downscaled (500 m)', True)
Map.centerObject(rockies, 9)
Map.turnOnInspector()
Map.view(True)

### The other variables downscale too

Wind is the fiddly one — it needs slope *in the wind direction*, curvature,
and a `region` to normalize against. Temperature, dewpoint and
precipitation are simpler, and they need no region: each depends only on
how far a pixel sits above **its own forecast cell's mean elevation**.

| | terrain term |
|---|---|
| `downscaleTemperature` | lapse rate × height above the cell mean |
| `downscaleDewpoint` | the same, at a shallower rate |
| `downscalePrecipitation` | orographic enhancement with height |

That reference is the part worth saying twice. A forecast's 2 m
temperature is a value for its cell's **mean** height, so measuring from
sea level instead would give every mountain a large, smooth, entirely
plausible cold bias.

Temperature and dewpoint use different rates deliberately: dewpoint falls
more slowly with height, so relative humidity rises going up. Downscale
temperature and leave dewpoint alone and you manufacture a mountain drier
than the forecast ever said — so do both, or neither.

**Still not new information.** A single lapse rate cannot express a valley
inversion, which is exactly what a clear winter night produces and exactly
when mountain temperature matters most. And the precipitation term has no
wind direction in it, so it enhances windward and lee slopes equally —
rain shadow is the largest orographic effect there is, and this does not
model it.

In [ ]:
Map.clearMap()

t_mtn = ee.Image(wx.getForecastData(past_start, past_end, 'gfs',
                                    variable='temperature_2m').first())
t_date = t_mtn.date().format('YYYY-MM-dd HH:mm').getInfo()
t_ds = wx.downscaleTemperature(t_mtn, scale=500)

# What it did, measured. dz is the height of each pixel above its own
# 28 km cell's mean -- the quantity the whole method runs on.
dz, _dem = wx._elevation_delta(t_mtn, None, 500)
d = dz.reduceRegion(ee.Reducer.percentile([2, 50, 98]), rockies, 500,
                    bestEffort=True, maxPixels=1e9).getInfo()
g = {k.rsplit('_p', 1)[1]: v for k, v in d.items()}
print(f'height above cell mean  p2={g["2"]:7.0f}  p50={g["50"]:7.0f}  '
      f'p98={g["98"]:7.0f} m')
print(f'so the lapse correction spans '
      f'{wx.DEFAULT_LAPSE_RATE * g["98"]:+.1f} to '
      f'{wx.DEFAULT_LAPSE_RATE * g["2"]:+.1f} C '
      f'at {wx.DEFAULT_LAPSE_RATE * 1000:.1f} C/km')

T_VIZ = {'min': -5, 'max': 30, 'palette': list(wx.TEMPERATURE_PALETTE),
         'yLabel': 'Temperature (C)'}
Map.addLayer(t_mtn.clip(rockies), T_VIZ,
             f'GFS temperature {t_date} - coarse (28 km)', False)
Map.addLayer(t_ds.clip(rockies), T_VIZ,
             f'GFS temperature {t_date} - downscaled (500 m)', True)

# Precipitation, same idea. Multiplicative rather than additive, and
# clamped: the formula has a pole at chi*dz = 1 past which it would
# return negative rain.
p_mtn = ee.Image(wx.getForecastData(past_start, past_end, 'gfs',
                                    variable='precipitation').first())
p_ds = wx.downscalePrecipitation(p_mtn, scale=500)
P_VIZ = {'min': 0, 'max': 3, 'palette': list(wx.PRECIP_PALETTE),
         'yLabel': 'Precipitation (mm/hr)'}
Map.addLayer(p_mtn.clip(rockies), P_VIZ,
             f'GFS precipitation {t_date} - coarse (28 km)', False)
Map.addLayer(p_ds.clip(rockies), P_VIZ,
             f'GFS precipitation {t_date} - downscaled (500 m)', False)

Map.centerObject(rockies, 9)
Map.turnOnInspector()
Map.view(True)

## Forecast time lapses

`Map.addTimeLapse` animates a collection rather than showing one instant.
For a forecast that is the natural view — a five-day run is a film, not a
frame.

Two things a forecast time lapse needs, both of which `getForecastData`
has already done:

* **`system:time_start` is the VALID time.** The viewer builds its frame
  list from the distinct dates actually present in the collection,
  formatted by `dateFormat`. WeatherNext stamps the *init* time there
  natively — one value for an entire run — so a raw collection collapses
  to a single frame.
* **One run.** A forward window that mixes runs makes the field jump
  wherever two runs disagree.

So `dateFormat` has to resolve your cadence, and `advanceInterval` is the
width of each frame's window. `'YYYY'` on hourly data gives you one frame
and no error.

**Thin the leads.** WeatherNext 3 publishes hourly out to 15 days; five
days of that is 120 frames, and every frame is its own tile layer.

> **This replaces the standalone `WeatherNextTimeLapse.py` example**,
> which had rotted. It was built on the WeatherNext **Graph** and **Gen**
> collections — both deprecated, and both stopped publishing on
> 2026-07-29. The replacements are `weathernext_2_0_0_mean`
> (deterministic) and `weathernext_2_0_0` (ensemble).
>
> Note the failure mode: a deprecated Earth Engine id keeps resolving, so
> nothing raised. The script found no recent run, printed *"usually
> transient — try again later"*, and exited successfully. It would have
> done that every day forever.

In [ ]:
Map.clearMap()

# Every 6 hours for 5 days: 20 frames. Hourly would be 120.
STEPS = list(range(6, 120 + 1, 6))

tl_model = 'weathernext' if HAS_WN else 'gfs'
tl_start = now.strftime('%Y-%m-%dT%H:%MZ')
tl_end   = (now + datetime.timedelta(days=5)).strftime('%Y-%m-%dT%H:%MZ')

def thin(ic):
    return ic.filter(ee.Filter.inList('lead_hours', STEPS))

# dateFormat resolves to the hour because the data is sub-daily;
# advanceInterval is the width of each frame's window, not the stride.
COMMON = {'dateFormat': 'YY-MM-dd HH', 'advanceInterval': 'hour',
          'canAreaChart': True,
          'areaChartParams': {'scale': 27830, 'minZoomSpecifiedScale': 5}}

temp = thin(wx.getForecastData(tl_start, tl_end, tl_model,
                               variable='temperature_2m'))
Map.addTimeLapse(temp,
                 {**COMMON, 'min': -20, 'max': 45,
                  'palette': list(wx.TEMPERATURE_PALETTE),
                  'legendLabelLeftAfter': 'C',
                  'legendLabelRightAfter': 'C'},
                 f'{tl_model}: Temperature (2 m)')

# windImage gives the query bands -- but it carries no time, so the stamp
# has to be put back. Without it every frame formats to the same date and
# the lapse is one frame long.
wind = thin(wx.getForecastData(tl_start, tl_end, tl_model))

def speed_kmh(img):
    img = ee.Image(img)
    return (wx.windImage(img, {'units': 'km/hr'}).select('speed')
              .set('system:time_start', img.get('system:time_start')))

Map.addTimeLapse(wind.map(speed_kmh),
                 {**COMMON, 'min': 0, 'max': 80,
                  'palette': list(wx.WIND_PALETTE),
                  'legendLabelLeftAfter': 'km/hr',
                  'legendLabelRightAfter': 'km/hr'},
                 f'{tl_model}: Wind speed (10 m)', False)

# mm/hr whichever model this is: GFS publishes an instantaneous rate
# and WeatherNext a one-hour accumulation, and those are the same
# quantity once converted. (ECMWF's is a running total since the run
# started, which is not -- it has its own variable.)
precip = thin(wx.getForecastData(tl_start, tl_end, tl_model,
                                 variable='precipitation'))
Map.addTimeLapse(precip,
                 {**COMMON, 'min': 0, 'max': 5,
                  'palette': list(wx.PRECIP_PALETTE),
                  'legendLabelLeftAfter': 'mm/hr',
                  'legendLabelRightAfter': 'mm/hr'},
                 f'{tl_model}: Precipitation', False)

print(f'{tl_model}: {temp.size().getInfo()} frames, '
      f'{tl_start[:13]} .. {tl_end[:13]}')

# No date in these names: every frame has its own valid time and the
# slider shows it. A single date on a time lapse would be wrong for all
# but one frame.
Map.turnOnInspector()
Map.view(True)

### Wind as a time lapse, particles and all

The cell above animates wind **speed** and loses the flow: it selects
`speed` off `windImage`, so the particles are gone and with them the only
thing on the map that shows direction.

`Map.addWindTimeLapse` animates the pair. Same two layers
`addWindLayer` adds -- the queryable raster and the particle canvas --
but each is a time lapse, so the slider scrubs the field while the
particles keep flowing through it.

Nothing new had to be invented for the frames. `windTiles` already packs
u and v into **one** RGB image (u in red, v in green), so a wind frame is
a single `ee.Image` by the time the viewer sees it; the encoder just gets
mapped over the collection.

Two things it handles that are easy to get wrong by hand:

* **`system:time_start` is put back.** `windImage` and `windTiles` both
  build a new image, and a new image carries no time. Unstamped, every
  frame formats to the same label and the lapse silently collapses to
  one frame -- which is exactly why the cell above re-stamps by hand.
* **`dateFormat` defaults to the hour**, not the year. `addTimeLapse`'s
  annual default is right for land cover and wrong for weather.

On the client, all the frames of one lapse are served by a **single**
particle overlay, grouped by the viewer's `timeLapseID` -- otherwise
nine frames would mean nine particle canvases stacked on the map, each
drawing its own flow. The decoded u/v **tiles** are cached per frame, so
scrubbing back and forth re-requests nothing after the first pass, and
the trails are never torn down: they carry on advecting through each new
field instead of resetting to a fresh scatter on every step.

Which frame the particles sample follows the lapse's **opacity**, not
its checkboxes -- a geeImage time lapse switches frames on the opacity
slider and leaves every frame's checkbox ticked.


In [ ]:
Map.clearMap()

# Six-hourly for two days: 9 frames (leads 0, 6, ... 48). Every frame is its own tile layer,
# so thin the leads -- hourly out to five days would be 120 of them.
wtl_start = now.strftime('%Y-%m-%dT%H:%MZ')
wtl_end   = (now + datetime.timedelta(days=2)).strftime('%Y-%m-%dT%H:%MZ')

wind_tl = wx.getForecastData(wtl_start, wtl_end, 'gfs').filter(
    ee.Filter.inList('lead_hours', list(range(0, 48 + 1, 6))))
print('frames:', wind_tl.size().getInfo())

speed_ic, tiles_ic = Map.addWindTimeLapse(
    wind_tl,
    {
        'units': 'kt',            # knots: what METAR, marine and windy.com use
        'min': 0, 'max': 60,      # stretch AND the particle speed bounds
        'particleSpeed': 0.5,     # px/frame per m/s, at the equator
        'particleTrailLength': 13,
    },
    'GFS wind')

# setCenter, not centerObject: a POINT has zero-area bounds, and
# centerObject fits the bounds -- so it lands at max zoom over one
# pixel of Nebraska instead of the 4 asked for.
Map.setCenter(-100.0, 40.0, 4)
Map.turnOnInspector()
Map.view()


### Real ensemble members

The ensemble percentiles above are WeatherNext 3's **pre-aggregated** ones, which is
all that product publishes — the members themselves are not in the
collection, so `p90 - p10` is the only spread available.

WeatherNext **2** is different: it publishes all 64 members as separate
images, tagged `ensemble_member`. That buys a genuine standard deviation,
and it is the only place you can get one. Worth the extra frames when the
question is *how confident is the forecast*, rather than *what is the
forecast*.

The cost is real — 64 images reduced per frame — so this thins harder,
to every 12 hours.

In [ ]:
# Temperature is Kelvin here, but a spread is a DIFFERENCE, and a
# difference in Kelvin is the same number in Celsius. No conversion.
Map.clearMap()
WN2 = 'projects/gcp-public-data-weathernext/assets/weathernext_2_0_0'
HOURS = list(range(12, 120 + 1, 12))

if HAS_WN:
    # Bound the run hunt. Scanning the whole archive for the newest run
    # is a reduce over every image ever published.
    recent = ee.ImageCollection(WN2).filter(ee.Filter.gt(
        'system:time_start',
        ee.Date(now - datetime.timedelta(hours=18)).millis()))
    run = ee.String(recent.aggregate_array('start_time').distinct().sort().get(-1))
    members = (recent.filter(ee.Filter.eq('start_time', run))
                     .filter(ee.Filter.inList('forecast_hour', HOURS))
                     .select(['2m_temperature']))

    def spread_frame(h):
        h = ee.Number(h)
        return (members.filter(ee.Filter.eq('forecast_hour', h))
                       .reduce(ee.Reducer.stdDev()).rename('spread')
                       .set('system:time_start',
                            ee.Date(run).advance(h, 'hour').millis(),
                            'forecast_hour', h))

    wn2_spread = ee.ImageCollection(ee.List(HOURS).map(spread_frame))
    Map.addTimeLapse(wn2_spread,
                     {**COMMON, 'min': 0, 'max': 4,
                      'palette': '000004,420a68,932667,dd513a,fca50a,fcffa4',
                      'legendLabelLeftAfter': 'C',
                      'legendLabelRightAfter': 'C',
                      'yLabel': 'Ensemble sd (C)'},
                     'WeatherNext 2: temperature spread (64 members)', False)

    # Measured, not asserted -- and over the DOMAIN, because a single
    # point is noisy enough to run backwards between consecutive frames.
    def _mean_sd(h):
        return (ee.Image(spread_frame(h))
                .reduceRegion(ee.Reducer.mean(), study_area, 50000,
                              bestEffort=True).getInfo()['spread'])

    lo_h, hi_h = _mean_sd(HOURS[0]), _mean_sd(HOURS[-1])
    print(f'mean ensemble sd over the West: '
          f'+{HOURS[0]}h {lo_h:.2f} C -> +{HOURS[-1]}h {hi_h:.2f} C '
          f'({hi_h / lo_h:.1f}x)')
    print('Confidence decays with lead time. That is the forecast telling')
    print('you how much to trust it, which a deterministic run cannot.')

Map.turnOnInspector()
Map.view()